In [21]:
# Fine tune modernbert base embed model with squad data to create a fine tuned embedding model
# which should be better than sentence transformer model

import pandas as pd
train_df_all = pd.read_csv("../data/squad_v2/train.csv")
val_df_all = pd.read_csv("../data/squad_v2/validation.csv")
len(train_df_all), len(val_df_all)

(130319, 11873)

In [24]:
# Filter those rows which are answerable, so as to take only those query and context pair for training
answer_col = "answers" 
pattern = r"""['"]text['"]\s*:\s*array\(\s*\[\s*['"]"""
train_df = train_df_all[
    train_df_all[answer_col].str.contains(pattern, regex=True, na=False)
].copy()
val_df = val_df_all[
    val_df_all[answer_col].str.contains(pattern, regex=True, na=False)
].copy()
len(train_df), len(val_df)

(86821, 5928)

In [25]:
# Sample N records
train_df_all.head(1000)

,id,title,context,question,answers
0,56be85543aeaaa14008c9063,Beyoncé,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,When did Beyonce start becoming popular?,"{'text': array(['in the late 1990s'], dtype=ob..."
1,56be85543aeaaa14008c9065,Beyoncé,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,What areas did Beyonce compete in when she was...,"{'text': array(['singing and dancing'], dtype=..."
2,56be85543aeaaa14008c9066,Beyoncé,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,When did Beyonce leave Destiny's Child and bec...,"{'text': array(['2003'], dtype=object), 'answe..."
3,56bf6b0f3aeaaa14008c9601,Beyoncé,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,In what city and state did Beyonce grow up?,"{'text': array(['Houston, Texas'], dtype=objec..."
4,56bf6b0f3aeaaa14008c9602,Beyoncé,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,In which decade did Beyonce become famous?,"{'text': array(['late 1990s'], dtype=object), ..."
...,...,...,...,...,...
995,56d319d459d6e41400146241,Frédéric_Chopin,"In Paris, Chopin encountered artists and other...",What position did the poet who Chopin knew in ...,{'text': array(['principal of the Polish Liter...
996,56cbf5106d243a140015ee18,Frédéric_Chopin,Two Polish friends in Paris were also to play ...,Which friend of Frédéric failed to achieve suc...,"{'text': array(['Julian Fontana'], dtype=objec..."
997,56cbf5106d243a140015ee19,Frédéric_Chopin,Two Polish friends in Paris were also to play ...,Who was Frédéric's trusted adviser while in Pa...,"{'text': array(['Albert Grzymała'], dtype=obje..."
998,56cbf5106d243a140015ee1a,Frédéric_Chopin,Two Polish friends in Paris were also to play ...,What familial role was Albert Grzymała compare...,"{'text': array(['elder brother'], dtype=object..."


In [26]:
# Taking 50 samples each in train and validation
from datasets import Dataset
train_dataset = Dataset.from_pandas(train_df[["question", "context"]].head(50).rename(columns={'question': 'query'}), preserve_index=False)
test_dataset =  Dataset.from_pandas(val_df[["question", "context"]].head(50).rename(columns={'question': 'query'}), preserve_index=False)
print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")
train_dataset


Train dataset size: 50
Test dataset size: 50


Dataset({
    features: ['query', 'context'],
    num_rows: 50
})

In [4]:
# Fine Tuning 

import torch
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers import SentenceTransformer

model_name = "../model/modernbert-embed-base"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(model_name, device=device)

loss = MultipleNegativesRankingLoss(model)


C:\Users\umada\AppData\Local\Temp\ipykernel_19488\2814646912.py:4: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers.losses import MultipleNegativesRankingLoss
Loading weights: 100%|██████████| 134/134 [00:00<00:00, 172.72it/s]


In [5]:
from sentence_transformers import SentenceTransformerTrainingArguments , SentenceTransformerTrainer

args = SentenceTransformerTrainingArguments(
    output_dir="../output",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=2e-5,
    warmup_steps=0.1,
    fp16=False, # i want to use fp32
    eval_strategy="steps",
    eval_steps=0.125,
    save_strategy="steps",
    save_steps=0,
    save_total_limit=2,
    logging_steps=1,
    report_to="none"
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    # tokenizer=model,
    loss=loss,
)

trainer.train()

H:\pythonProject\natural_language_processing\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss
4,0.693147,0.309183
8,0.357895,0.290081
12,0.093835,0.283628
16,0.019016,0.276787
20,0.212204,0.274234
24,0.005946,0.271322
25,0.454624,0.271116


Writing model shards: 100%|██████████| 1/1 [00:04<00:00,  4.47s/it]


TrainOutput(global_step=25, training_loss=0.40234856605529784, metrics={'train_runtime': 956.9818, 'train_samples_per_second': 0.052, 'train_steps_per_second': 0.026, 'total_flos': 0.0, 'train_loss': 0.40234856605529784, 'epoch': 1.0})